In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil
from pathlib import Path

DRIVE_REPO = Path('/content/drive/MyDrive/movie-recomender-search')
LOCAL_REPO = Path('/content/movie-recomender-search')

if LOCAL_REPO.exists():
    shutil.rmtree(LOCAL_REPO)

shutil.copytree(DRIVE_REPO, LOCAL_REPO)
os.chdir(LOCAL_REPO)

print('Working directory:', Path.cwd())
print('Drive repo:', DRIVE_REPO)
print('Local repo:', LOCAL_REPO)

# 02 - Matrix Factorization

Notebook baseline để huấn luyện `Matrix Factorization` trên MovieLens 1M.

Mục tiêu:
- tạo baseline học máy truyền thống
- đánh giá trên validation và test
- lưu metric, bảng kết quả, model artifact và figure để so sánh với NeuMF

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import FIGURES_DIR, METRICS_DIR, MODELS_DIR, RAW_DIR
from src.data.load_movielens import load_ratings
from AI.src.data.split import random_split_explicit
from AI.src.models.matrix_factorization import MFConfig
from AI.src.training.train_mf import save_metrics, save_mf_artifacts, train_matrix_factorization
from AI.src.utils.seed import set_seed

set_seed(42)
print(f'Project root: {PROJECT_ROOT}')
print(f'Data dir: {RAW_DIR}')
print(f'Metrics dir: {METRICS_DIR}')
print(f'Models dir: {MODELS_DIR}')
print(f'Figures dir: {FIGURES_DIR}')

In [ ]:
ratings = load_ratings()
summary = {
    'n_rows': len(ratings),
    'n_users': int(ratings['user_id'].nunique()),
    'n_items': int(ratings['movie_id'].nunique()),
    'rating_min': int(ratings['rating'].min()),
    'rating_max': int(ratings['rating'].max()),
}
pd.Series(summary)

In [ ]:
train_df, val_df, test_df = random_split_explicit(
    ratings,
    train_frac=0.8,
    val_frac=0.1,
    random_state=42,
)

split_summary = pd.DataFrame(
    {
        'split': ['train', 'validation', 'test'],
        'rows': [len(train_df), len(val_df), len(test_df)],
        'users': [train_df['user_id'].nunique(), val_df['user_id'].nunique(), test_df['user_id'].nunique()],
        'items': [train_df['movie_id'].nunique(), val_df['movie_id'].nunique(), test_df['movie_id'].nunique()],
    }
)
split_summary

In [ ]:
mf_config = MFConfig(
    num_factors=32,
    learning_rate=0.01,
    reg=0.02,
    epochs=20,
    random_state=42,
    show_progress=True,
)
mf_config

In [ ]:
model, metrics, user_to_index, item_to_index = train_matrix_factorization(
    train_df,
    val_df,
    test_ratings=test_df,
    config=mf_config,
)

pd.Series(
    {
        'best_epoch': metrics['best_epoch'],
        'best_val_rmse': metrics['best_val_rmse'],
        'test_rmse': metrics['test_metrics']['rmse'],
        'test_mae': metrics['test_metrics']['mae'],
    }
)

In [ ]:
results_table = pd.DataFrame(
    [
        {'split': 'validation', **metrics['val_metrics']},
        {'split': 'test', **metrics['test_metrics']},
    ]
)
results_table

In [ ]:
metrics_path = save_metrics('mf_metrics.json', metrics)
model_path = save_mf_artifacts('mf_model.npz', model, user_to_index, item_to_index)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
results_table_path = METRICS_DIR / 'mf_results_table.csv'
results_table.to_csv(results_table_path, index=False)

print('Saved metrics      ->', metrics_path)
print('Saved model        ->', model_path)
print('Saved results table->', results_table_path)

In [ ]:
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
figure_path = FIGURES_DIR / 'mf_training_curve.png'

plt.figure(figsize=(10, 4))
plt.plot(metrics['train_history_rmse'], marker='o', label='train rmse')
plt.plot(metrics['val_history_rmse'], marker='s', label='val rmse')
plt.axvline(metrics['best_epoch'] - 1, color='red', linestyle='--', alpha=0.6, label='best epoch')
plt.title('MF Training and Validation RMSE by Epoch')
plt.xlabel('Epoch')
plt.ylabel('RMSE')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(figure_path, dpi=150, bbox_inches='tight')
plt.show()
print('Saved figure       ->', figure_path)